In [ ]:
Laila Said: laila_204265

Leen Ghoneem: leen_161577

Adnan Al-As'ad: adnan_189704

In [ ]:
# basic stuff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, silhouette_score)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering

np.random.seed(42)
print("imports done")

# colab might be blocked behind a 403 Forbidden error, this cell is for mitigating that
import sklearn.datasets
from sklearn.utils import Bunch

def fake_fetch(as_frame=True):
    url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
    d = pd.read_csv(url)
    d['total_bedrooms'] = d['total_bedrooms'].fillna(d['total_bedrooms'].median())

    data = pd.DataFrame({
        'MedInc': d['median_income'],
        'HouseAge': d['housing_median_age'],
        'AveRooms': d['total_rooms'] / d['households'],
        'AveBedrms': d['total_bedrooms'] / d['households'],
        'Population': d['population'],
        'AveOccup': d['population'] / d['households'],
        'Latitude': d['latitude'],
        'Longitude': d['longitude']
    })

    target = d['median_house_value'] / 100000.0

    frame = data.copy()
    frame['MedHouseVal'] = target

    return Bunch(data=data.values, target=target.values, frame=frame, feature_names=list(data.columns))

sklearn.datasets.fetch_california_housing = fake_fetch
from sklearn.datasets import fetch_california_housing

In [ ]:
# Load the dataset with fetch_california_housing(as_frame=True)
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# rename target to match the assignment
df.rename(columns={'MedHouseVal': 'MedHouseValue'}, inplace=True)

print(df.shape)
df.info()
df.isnull().sum()
df.head()
df.describe()

# split features and target
X = df.drop(columns=['MedHouseValue'])
y = df['MedHouseValue']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

<div dir="rtl">
<b>ملخص الجزء الأول.</b> تحتوي البيانات على 20,640 صفاً و8 خصائص، وهي: MedInc و HouseAge و AveRooms و AveBedrms و Population و AveOccup و Latitude و Longitude. المتغير الهدف هو MedHouseValue (قيمة المنزل بوحدة 100,000 دولار). تم تقسيم البيانات بنسبة 80% للتدريب و20% للاختبار.
</div>

In [ ]:
# baseline linear regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R2:", round(r2, 4))

In [ ]:
# ridge and lasso, need scaling first
scal = StandardScaler()

X_train_2 = scal.fit_transform(X_train)
X_test_2 = scal.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_2, y_train)
y_pred_ridge = ridge.predict(X_test_2)

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_2, y_train)
y_pred_lasso = lasso.predict(X_test_2)

print("Ridge:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 4),
      round(r2_score(y_test, y_pred_ridge), 4))
print("Lasso:", round(np.sqrt(mean_squared_error(y_test, y_pred_lasso)), 4),
      round(r2_score(y_test, y_pred_lasso), 4))

In [ ]:
# polynomial features for MedInc and HouseAge only
poly_cols = ['MedInc', 'HouseAge']

pre = ColumnTransformer([('poly', PolynomialFeatures(degree=2, include_bias=False), poly_cols)],
                        remainder='passthrough')

poly_model = Pipeline([('pre', pre), ('scaler', StandardScaler()), ('lin', LinearRegression())])
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)

print("Poly RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_poly)), 4))
print("Poly R2:", round(r2_score(y_test, y_pred_poly), 4))

In [ ]:
# which features matter most
coefs = pd.DataFrame({'Feature': X_train.columns, 'Coef': lin_reg.coef_})
coefs['AbsCoef'] = coefs['Coef'].abs()
coefs = coefs.sort_values('AbsCoef', ascending=False)
coefs = coefs.reset_index(drop=True)
coefs

<div dir="rtl">
<b>ملخص الجزء الثاني.</b> تم اختبار أربعة نماذج للانحدار. حقق نموذج Linear Regression (خط الأساس) قيم RMSE = 0.7456 و R² = 0.5758. أظهر نموذج Ridge نتيجة مماثلة تقريباً لأن التنظيم (Regularization) كان ضعيفاً. وكان Lasso الأسوأ (R² = 0.4814) لأن العقوبة كانت مفرطة وألغت معاملات مفيدة.
<br><br>
تمت إضافة متغيرات Polynomial من الدرجة الثانية لـ MedInc و HouseAge. تحسنت النتيجة قليلاً (RMSE = 0.7413 و R² = 0.5806). هذا يدل على وجود علاقة غير خطية لكنها ليست قوية، والعلاقة الخطية كافية.
<br><br>
عند فحص معاملات النموذج، ظهرت AveBedrms كأكبر قيمة (0.783). لكن هذا لا يعني أنها الأهم، بسبب مشكلة التعدد الخطي (Multicollinearity) مع AveRooms، مما جعل المعاملات غير مستقرة. المتغيرات الأهم فعلياً هي MedInc والموقع (Latitude و Longitude). أفضل نموذج كان Polynomial ولكن الفرق بسيط جداً عن الخطي.
</div>